# Need many threads for this not to crash

In [1]:
here::i_am("rna/mapping/run/mnn/mapping_mnn.R")

# Load default settings
source(here::here("settings.R"))
source(here::here("utils.R"))

# Load packages
suppressPackageStartupMessages(library(scran))
suppressPackageStartupMessages(library(scater))
suppressPackageStartupMessages(library(batchelor))

######################
## Define arguments ##
######################

# p <- ArgumentParser(description='')
# p$add_argument('--atlas_stages',    type="character",   nargs='+',  help='Atlas stage(s)')
# p$add_argument('--query_samples',   type="character",   nargs='+',  help='Query batch(es)')
# p$add_argument('--query_sce',       type="character",               help='SingleCellExperiment file for the query')
# p$add_argument('--atlas_sce',       type="character",               help='SingleCellExperiment file for the atlas')
# p$add_argument('--query_metadata',  type="character",               help='metadata file for the query')
# p$add_argument('--atlas_metadata',  type="character",               help='metadata file for the atlas')
# p$add_argument('--npcs',            type="integer",                 help='Number of principal components')
# p$add_argument('--n_neighbours',    type="integer",                 help='Number of neighbours')
# p$add_argument('--use_marker_genes',action = "store_true",          help='Use marker genes?')
# p$add_argument('--cosine_normalisation',      action = "store_true",          help='Use cosine normalisation?')
# p$add_argument('--test',            action = "store_true",          help='Testing mode')
# p$add_argument('--outfile',          type="character",               help='Output file')
# args <- p$parse_args(commandArgs(TRUE))

# #####################
## Define settings ##
#####################

# I/O
io$path2atlas <- io$atlas.basedir
io$path2query <- io$basedir


# Load mapping functions
source(here::here("rna/mapping/run/mnn/mapping_functions.R"))


## START TEST ##
args = list()
args$atlas_stages <- c("E6.5","E6.75","E7.0","E7.25","E7.5","E7.75","E8.0","E8.25","E8.5")
args$query_samples <- opts$samples
args$query_sce <- io$rna.sce
args$query_sce <- paste0(io$basedir,"/processed/rna/SingleCellExperiment.rds")
args$atlas_sce <- io$rna.atlas.sce
args$query_metadata <- file.path(io$basedir,"results/rna/mapping/sample_metadata_after_mapping.txt.gz")
args$atlas_metadata <- io$rna.atlas.metadata
args$test <- FALSE
args$npcs <- 50
args$n_neighbours <- 25
args$use_marker_genes <- FALSE
args$cosine_normalisation <- FALSE
args$outfolder <- paste0(io$basedir,"/code/rna/mapping/scVI_test")
## END TEST ##

if (isTRUE(args$test)) print("Test mode activated...")


here() starts at /rds/project/rds-SDzz0CATGms/users/bt392/09_Eomes_invitro_blood/code

Warning message:
“package ‘argparse’ was built under R version 4.1.3”


In [2]:

################
## Load query ##
################

# Load cell metadata
meta_query <- fread(args$query_metadata) %>% 
  .[pass_rnaQC==TRUE & doublet_call==FALSE & sample%in%args$query_samples]
if (isTRUE(args$test)) meta_query <- head(meta_query,n=1000)

# Load SingleCellExperiment
sce_query <- load_SingleCellExperiment(args$query_sce, cells = meta_query$cell, remove_non_expressed_genes = TRUE)

# Update colData
tmp <- meta_query %>% .[cell%in%colnames(sce_query)] %>% setkey(cell) %>% .[colnames(sce_query)]
stopifnot(tmp$cell == colnames(sce_query))
colData(sce_query) <- tmp %>% as.data.frame %>% tibble::column_to_rownames("cell") %>%
  .[colnames(sce_query),] %>% DataFrame()

In [3]:
################
## Load atlas ##
################

# Load cell metadata
meta_atlas <- fread(args$atlas_metadata) %>%
  .[stripped==F & doublet==F & stage%in%args$atlas_stages] %>%
  .[,sample:=factor(sample)]

# Filter
if (isTRUE(args$test)) meta_atlas <- head(meta_atlas,n=38346)

# Load SingleCellExperiment
sce_atlas <- load_SingleCellExperiment(args$atlas_sce, normalise = TRUE, cells = meta_atlas$cell, remove_non_expressed_genes = TRUE)

# Update colData
tmp <- meta_atlas %>% .[cell%in%colnames(sce_atlas)] %>% setkey(cell) %>% .[colnames(sce_atlas)]
stopifnot(tmp$cell == colnames(sce_atlas))
colData(sce_atlas) <- tmp %>% as.data.frame %>% tibble::column_to_rownames("cell") %>%
  .[colnames(sce_atlas),] %>% DataFrame()


In [4]:
sce_atlas
sce_query

class: SingleCellExperiment 
dim: 19379 108857 
metadata(1): log.exprs.offset
assays(2): counts logcounts
rownames(19379): ENSMUSG00000051951 ENSMUSG00000025900 ...
  ENSMUSG00000063897 ENSMUSG00000095742
rowData names(0):
colnames(108857): cell_1 cell_10 ... cell_99998 cell_99999
colData names(11): barcode sample ... nFeature_RNA nCount_RNA
reducedDimNames(0):
mainExpName: NULL
altExpNames(0):

class: SingleCellExperiment 
dim: 23004 38346 
metadata(0):
assays(1): counts
rownames(23004): Xkr4 Gm1992 ... CAAA01147332.1 AC149090.1
rowData names(0):
colnames(38346): 1A_Eo_DEG_G9_day3#AAACAGCCAGCAAGAT-1
  1A_Eo_DEG_G9_day3#AAACAGCCAGCACCAT-1 ...
  rv_eo_deg_day4_dtag#TTTGTGTTCATTTGTC-1
  rv_eo_deg_day4_dtag#TTTGTTGGTACTTAGG-1
colData names(17): barcode sample ... day_celltype celltype_genotype
reducedDimNames(0):
mainExpName: RNA
altExpNames(0):

In [5]:
#############
## Prepare ## 
#############

# Rename ensemble IDs to gene names in the atlas
gene_metadata <- fread(io$gene_metadata) %>% .[,c("chr","ens_id","symbol")] %>%
  .[symbol!="" & ens_id%in%rownames(sce_atlas)] %>%
  .[!duplicated(symbol)]

sce_atlas <- sce_atlas[rownames(sce_atlas)%in%gene_metadata$ens_id,]
foo <- gene_metadata$symbol; names(foo) <- gene_metadata$ens_id
rownames(sce_atlas) <- foo[rownames(sce_atlas)]

# Sanity checks
stopifnot(sum(is.na(rownames(sce_atlas)))==0)
stopifnot(sum(duplicated(rownames(sce_atlas)))==0)

stopifnot(TRUE %in% (meta_query$cell %in% meta_atlas$cell)==FALSE)

In [6]:
#####################
## Define gene set ##
#####################

# Intersect genes
genes.intersect <- intersect(rownames(sce_query), rownames(sce_atlas))

# Filter some genes manually
genes.intersect <- genes.intersect[grep("^Rik|Rik$|^mt-|^Rps-|^Rpl-|^Gm",genes.intersect,invert=T)]
genes.intersect <- genes.intersect[!genes.intersect=="Xist"]
genes.intersect <- genes.intersect[!genes.intersect%in%gene_metadata[chr=="chrY",symbol]]

# Subset SingleCellExperiment objects
sce_query  <- sce_query[genes.intersect,]
sce_atlas <- sce_atlas[genes.intersect,]

In [7]:
meta_atlas = as.data.table(colData(sce_atlas), keep.rownames=T) %>%
    setnames('rn', 'cell') %>%
    .[,c('cell', 'sample', 'stage', 'celltype')] %>%
    .[,origin:='atlas']

colData(sce_atlas) <- meta_atlas %>% as.data.frame %>% tibble::column_to_rownames("cell") %>%
  .[colnames(sce_atlas),] %>% DataFrame()

Warning message in .local(x, row.names, optional, ...):
“Arguments in '...' ignored”


In [8]:
meta_query = as.data.table(colData(sce_query), keep.rownames=T) %>%
    setnames('rn', 'cell') %>% setnames('day', 'stage') %>%
    .[,c('cell', 'sample', 'stage', 'celltype')] %>% 
    .[,origin:='query']

colData(sce_query) <- meta_query %>% as.data.frame %>% tibble::column_to_rownames("cell") %>%
  .[colnames(sce_query),] %>% DataFrame()

Warning message in .local(x, row.names, optional, ...):
“Arguments in '...' ignored”


In [9]:
# sceasy::convertFormat(sce_atlas, from="sce", 
#                       to="anndata",
#                       outFile=file.path(args$outfolder, 'sce_atlas.h5ad'))
# sceasy::convertFormat(sce_query, from="sce", 
#                       to="anndata",
#                       outFile=file.path(args$outfolder, 'sce_query.h5ad'))

In [10]:
big_meta = rbind(meta_query, meta_atlas)

In [11]:
counts_query = counts(sce_query)
counts_atlas = counts(sce_atlas)

In [12]:
big_counts = cbind(counts_query, counts_atlas)

In [13]:
big_sce = SingleCellExperiment(list(counts=big_counts))

In [14]:
colData(big_sce) <- big_meta %>% as.data.frame %>% tibble::column_to_rownames("cell") %>%
  .[colnames(big_sce),] %>% DataFrame()

In [15]:
big_sce

class: SingleCellExperiment 
dim: 15403 147203 
metadata(0):
assays(1): counts
rownames(15403): Xkr4 Rp1 ... Hccs Mid1
rowData names(0):
colnames(147203): 1A_Eo_DEG_G9_day3#AAACAGCCAGCAAGAT-1
  1A_Eo_DEG_G9_day3#AAACAGCCAGCACCAT-1 ... cell_99998 cell_99999
colData names(4): sample stage celltype origin
reducedDimNames(0):
mainExpName: NULL
altExpNames(0):

In [16]:
sceasy::convertFormat(big_sce, from="sce", 
                      to="anndata",
                      outFile=file.path(args$outfolder, 'anndata.h5ad'))

AnnData object with n_obs × n_vars = 147203 × 15403
    obs: 'sample', 'stage', 'celltype', 'origin'
    var: 'name'

In [17]:
file.path(args$outfolder, 'anndata.h5ad')

[1] "/rds/project/rds-SDzz0CATGms/users/bt392/09_Eomes_invitro_blood/code/rna/mapping/scVI_test/anndata.h5ad"